In [ ]:
!pip install langchain-community pypdf


In [ ]:
!pip install -q \
transformers==4.53.3 \
sentence-transformers==5.1.0 \
huggingface_hub==0.34.4 \
accelerate \
sentencepiece

In [ ]:
! pip install qdrant-client

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader=PyPDFLoader("BOOK PATH")
docs=loader.load()

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunk=splitter.split_documents(docs)

In [ ]:
import re

for doc in chunk:
    text = doc.page_content

    text = re.sub(r"EBSCOhost.*", "", text)

    
    text = re.sub(r"\n?\d+\s+Representing.*", "", text)

   
    text = re.sub(r"\n{2,}", "\n", text)

    doc.page_content = text.strip()

In [ ]:
new_chunk=chunk
type(new_chunk[0])

In [ ]:
from sentence_transformers import SentenceTransformer
bert_model=SentenceTransformer("BAAI/bge-small-en-v1.5")

In [ ]:
def doc_embed(new_chunk):
    
    
    document=[doc.page_content for doc in new_chunk]
    document_embeddings=bert_model.encode(
           document,
            convert_to_numpy=True
    )
    return document_embeddings

In [ ]:
def vector_db(new_chunk,document_embeddings):
    from qdrant_client import QdrantClient
    from qdrant_client.models import PointStruct
    from qdrant_client.models import Distance,models
    client=QdrantClient(":memory:")
    
    client.create_collection(
    collection_name="nlp_rag",
    vectors_config=models.VectorParams(size=384,distance=Distance.COSINE)    
    )

    
    points=[];
    for  i, (chunk,embed) in enumerate(zip(new_chunk,document_embeddings)):
            points.append(
                PointStruct(
                    id=i,
                    vector=embed.tolist(),
                    payload={
                    "text":chunk.page_content
                        }    
                )
        
        
        )


    
    client.upsert(
        collection_name="nlp_rag",
        points=points
    )
    return client
    

In [ ]:
from sentence_transformers import CrossEncoder
reranked=CrossEncoder(
"cross-encoder/ms-marco-MiniLM-L-6-v2"
    )

In [ ]:
def user_embed(user_query,cl,reranked):
    
    user_embeddings=bert_model.encode(
        user_query
    )
    # from qdrant_client_models import search
    results=cl.query_points(
        collection_name="nlp_rag",
        query=user_embeddings.tolist(),
        limit=30
    )

    
    
    points = results.points
    
    rerank_doc=[[user_query,point.payload["text"]] for point in points] 
    
    scores=reranked.predict(rerank_doc)
    
    ranked_docs = sorted(
    zip(rerank_doc,scores ),
    key=lambda x: x[1],
    reverse=True
    )
    
    for i, (doc, score) in enumerate(ranked_docs[:5]):
        print(f"\nRank {i+1} | Score: {score:.2f}")
        print(doc[1][:500])
        print("-" * 80)
    
    context = "\n".join(
    doc[1]
    for doc, score in ranked_docs[:5]
    )
    
    return context



    

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
model_name = "microsoft/Phi-3-mini-4k-instruct"
    
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
    )

In [ ]:
def llm(context, text, model):
    prompt = f"""
You are a helpful assistant.

Answer the question using ONLY the provided context.

You may summarize, compare, or combine information from different parts of the context.

If the context does not contain enough information to answer the question, reply:

I don't know.

Context:
{context}

Question:
{text}

Answer:
"""
    print("=="*100)
    print(context)
    print("=="*100)
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1
    )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    return response

In [ ]:
document_embed=doc_embed(new_chunk)

In [ ]:
while True:

    query = input("Ask: ")

    if query.lower() == "exit":
        break
    cl=vector_db(new_chunk,document_embed)

    con=user_embed(query,cl,reranked)
    

    resp=llm(con,query,model)
    

    

    print("\nAnswer:\n")
    print(resp)